# IMLLS — Thesis Analysis Notebook
Generates all charts and statistics needed for the diploma.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_theme(style='darkgrid', palette='muted')
LOGS_DIR = Path('logs')

# Load all user logs
dfs = []
for f in LOGS_DIR.glob('*.csv'):
    dfs.append(pd.read_csv(f))
log = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f'Total interactions: {len(log)}')
log.head()

In [ ]:
# ── Chart 1: Similarity score by step ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
step_scores = log.groupby('step')['similarity'].agg(['mean', 'std'])
ax.bar(step_scores.index, step_scores['mean'],
       yerr=step_scores['std'], capsize=4,
       color=['#5060a0' if s < 7 else '#50a070' for s in step_scores.index])
ax.axhline(0.82, color='orange', linestyle='--', label='Pass threshold (82%)')
ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Mean Similarity Score', fontsize=12)
ax.set_title('Pronunciation Accuracy by Step', fontsize=14)
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig('figures/similarity_by_step.png', dpi=150)
plt.show()

In [ ]:
# ── Chart 2: Learning curve ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
rolling = log['similarity'].rolling(window=10, min_periods=1).mean()
ax.plot(rolling, color='#7080d0', linewidth=2)
ax.fill_between(range(len(rolling)), rolling, alpha=0.15, color='#7080d0')
ax.axhline(0.82, color='orange', linestyle='--', alpha=0.7, label='Pass threshold')
ax.set_xlabel('Interaction #')
ax.set_ylabel('Similarity (rolling avg 10)')
ax.set_title('Learning Curve')
ax.legend()
plt.tight_layout()
plt.savefig('figures/learning_curve.png', dpi=150)
plt.show()

In [ ]:
# ── Chart 3: Cold start vs Adaptive ──────────────────────────────────────
cold     = log[log['mode'] == 'cold_start']['similarity'].dropna()
adaptive = log[log['mode'] == 'adaptive']['similarity'].dropna()

if len(cold) > 1 and len(adaptive) > 1:
    t_stat, p_value = stats.ttest_ind(cold, adaptive)
    print(f'Cold start mean:  {cold.mean():.3f}')
    print(f'Adaptive mean:    {adaptive.mean():.3f}')
    print(f't-statistic:      {t_stat:.3f}')
    print(f'p-value:          {p_value:.4f}')
    print(f'Significant:      {"YES" if p_value < 0.05 else "NO"} (α=0.05)')

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.boxplot([cold, adaptive], labels=['Cold Start', 'Adaptive'])
    ax.set_ylabel('Similarity Score')
    ax.set_title(f'Cold Start vs Adaptive  (p={p_value:.4f})')
    plt.tight_layout()
    plt.savefig('figures/cold_vs_adaptive.png', dpi=150)
    plt.show()
else:
    print('Not enough data yet — need interactions in both modes.')

In [ ]:
# ── Chart 4: Pass rate by step ────────────────────────────────────────────
pass_rate = log.groupby('step')['success'].mean() * 100
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(pass_rate.index, pass_rate.values,
       color=['#a05050' if v < 70 else '#50a050' for v in pass_rate.values])
ax.set_xlabel('Step')
ax.set_ylabel('Pass Rate (%)')
ax.set_title('Pass Rate by Step')
ax.set_ylim(0, 105)
for i, v in pass_rate.items():
    ax.text(i, v + 1.5, f'{v:.0f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('figures/pass_rate_by_step.png', dpi=150)
plt.show()